# BEAM Benchmark — Agent-Graph-Memory

Runs the BEAM long-term memory benchmark using Fireworks.ai as the LLM backend.

**No GPU needed** — this notebook only calls the Fireworks API remotely.

## Instructions
1. Run cells top-to-bottom
2. Cell 3 mounts Google Drive — results will be saved there so they persist
3. Cell 6 runs the benchmark (this is the long-running cell)
4. Cell 7 scores the results
5. Keep this tab open while the benchmark runs

**Tiers:** `100K` (20 chats), `500K` (35 chats), `1M` (35 chats)

**Estimated time (100K, 4 samples):** 4–8 hours

## 1. Clone repo & install dependencies

In [ ]:
import os

# Clone the repo (fresh clone each session)
if not os.path.exists('/content/Agentic-Graph-Memory'):
    !git clone https://github.com/nmokaria27/Agentic-Graph-Memory.git /content/Agentic-Graph-Memory

%cd /content/Agentic-Graph-Memory

# Checkout the feature branch
!git checkout feat/vector-index-and-qa-improvements

# Install the package + benchmark dependencies
!pip install -e . -q
!pip install datasets json_repair sentence_transformers scipy -q

print("\n=== Dependencies installed ===")

## 2. Configure `.env` with Fireworks credentials

In [ ]:
# Pull the Fireworks API key from a Colab Secret (left sidebar → 🔑 Secrets,
# add a secret named FIREWORKS_API_KEY). Falls back to a hidden prompt.
# This keeps the key OUT of the notebook so it is safe to commit/push.
try:
    from google.colab import userdata
    FIREWORKS_API_KEY = userdata.get("FIREWORKS_API_KEY")
except Exception:
    FIREWORKS_API_KEY = None
if not FIREWORKS_API_KEY:
    import getpass
    FIREWORKS_API_KEY = getpass.getpass("Fireworks API key: ")

env = f"""# ── Fireworks.ai backend ───────────────────────────────────────────
LLM_BACKEND=vllm
VLLM_BASE_URL=https://api.fireworks.ai/inference/v1
VLLM_API_KEY={FIREWORKS_API_KEY}
LLM_DEFAULT_MODEL=accounts/fireworks/models/deepseek-v4-flash

EMBEDDING_BASE_URL=https://api.fireworks.ai/inference/v1
EMBEDDING_API_KEY={FIREWORKS_API_KEY}
EMBEDDING_MODEL=accounts/fireworks/models/qwen3-embedding-8b

# ── Concurrency (lower = fewer 429s) ──────────────────────────────
ENTITY_EXTRACT_WORKERS=3
RELATION_EXTRACT_WORKERS=3
# Wait longer between retries so a 429 doesn't immediately re-burst.
LLM_RETRY_BACKOFF=5

# ── Judge model for Phase 2 scoring ───────────────────────────────
# Using v4-flash for judge too. Change to accounts/fireworks/models/deepseek-v3
# for higher-quality judging.
BEAM_JUDGE_MODEL=accounts/fireworks/models/deepseek-v4-flash
"""
with open(".env", "w") as f:
    f.write(env)

print("=== .env written (key sourced from Colab secret / prompt) ===")

## 3. Mount Google Drive (for persistent result storage)

Results will be saved to `MyDrive/beam_results/` so they survive session disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create output directories on Google Drive
!mkdir -p /content/drive/MyDrive/beam_results/kg_cache
!mkdir -p /content/drive/MyDrive/beam_results/responses
!mkdir -p /content/drive/MyDrive/beam_results/scores

# Also create local dirs (symlinks to Drive for the script)
!mkdir -p evaluation/results/beam_kg_cache
!mkdir -p evaluation/results

print("=== Google Drive mounted ===")
print("Results will be saved to: /content/drive/MyDrive/beam_results/")

## 4. Quick connectivity test

Verify Fireworks API is reachable from Colab.

In [ ]:
import requests

resp = requests.get(
    "https://api.fireworks.ai/inference/v1/models",
    headers={"Authorization": f"Bearer {FIREWORKS_API_KEY}"},
    timeout=10
)
if resp.status_code == 200:
    models = resp.json().get("data", [])
    print(f"✅ Fireworks API reachable — {len(models)} models available")
    model_ids = [m["id"] for m in models]
    for target in ["accounts/fireworks/models/deepseek-v4-flash", "accounts/fireworks/models/qwen3-embedding-8b"]:
        status = "✅" if target in model_ids else "⚠️ (not in /models list — usually fine for serverless)"
        print(f"  {status} {target}")
else:
    print(f"❌ Fireworks API error: {resp.status_code} — {resp.text[:200]}")

## 5. Configure run parameters

Edit these before running the benchmark.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  EDIT THESE PARAMETERS                                        ║
# ╚══════════════════════════════════════════════════════════════╝

TIER = "100K"              # Options: 100K, 500K, 1M
MAX_SAMPLES = 4            # Number of conversations (-1 = all). 4 = ~4-8 hrs
MAX_QUESTIONS = -1         # Questions per type per chat (-1 = all)
CHUNK_SIZE = 3000          # Chunk size in chars (3000 recommended for 100K)
RETRIEVAL = "hybrid"       # Options: hybrid, graph_completion, dense, lexical, chunk
MAX_CONTEXT_CHARS = ""     # Truncate conversation (empty = no truncation)

# Derived paths — KG cache lives on DRIVE so the expensive builds survive a
# Colab disconnect (local /content is wiped on reconnect). Resume then skips
# already-built KGs instead of rebuilding from scratch.
KG_DIR = f"/content/drive/MyDrive/beam_results/kg_cache/{TIER}"
RESPONSES_PATH = f"/content/drive/MyDrive/beam_results/responses/beam_{TIER}_responses.json"
SCORES_PATH = f"/content/drive/MyDrive/beam_results/scores/beam_{TIER}_scores.json"

# Build the command
cmd = f"""
python evaluation/BEAM/run_eval.py \\
    --tier {TIER} \\
    --max-samples {MAX_SAMPLES} \\
    --max-questions {MAX_QUESTIONS} \\
    --chunk-size {CHUNK_SIZE} \\
    --retrieval {RETRIEVAL} \\
    --save-kg-dir {KG_DIR} \\
    --output {RESPONSES_PATH}
"""
if MAX_CONTEXT_CHARS:
    cmd += f"    --max-context-chars {MAX_CONTEXT_CHARS} \\
"

print("Run command:")
print(cmd)
print(f"\nKG cache  → {KG_DIR}")
print(f"Responses → {RESPONSES_PATH}")
print(f"Scores     → {SCORES_PATH}")

## 6. Run BEAM Phase 1 — Build KG + Answer Questions

**This is the long-running cell.** Keep the tab open. Output streams below.

If Colab disconnects, results saved so far are in Google Drive. Re-run with `--load-kg-dir` to resume.

In [ ]:
import time

start = time.time()
print(f"Starting BEAM Phase 1 at {time.strftime('%H:%M:%S')}")
print(f"Tier: {TIER} | Samples: {MAX_SAMPLES} | Retrieval: {RETRIEVAL}")
print("=" * 60)

!{cmd.strip()}

elapsed = time.time() - start
hours = elapsed / 3600
print(f"\n{'=' * 60}")
print(f"Phase 1 complete in {hours:.1f} hours")
print(f"Responses saved to: {RESPONSES_PATH}")

## 7. Run BEAM Phase 2 — Score with LLM Judge

Uses an LLM-as-judge to score each answer. Takes ~2-5 hours for 4 samples.

If interrupted, re-run with `--resume` to skip already-scored questions.

In [ ]:
import time

start = time.time()
print(f"Starting BEAM Phase 2 (Scoring) at {time.strftime('%H:%M:%S')}")
print("=" * 60)

!python evaluation/BEAM/score.py \
    --responses {RESPONSES_PATH} \
    --output {SCORES_PATH} \
    --resume

elapsed = time.time() - start
print(f"\n{'=' * 60}")
print(f"Phase 2 complete in {elapsed/3600:.1f} hours")
print(f"Scores saved to: {SCORES_PATH}")

## 8. View aggregate results

In [ ]:
import json
from pathlib import Path

scores_path = Path(SCORES_PATH)
if scores_path.exists():
    with open(scores_path) as f:
        data = json.load(f)
    
    print("=" * 60)
    print("BEAM Aggregate Scores")
    print("=" * 60)
    print(json.dumps(data.get("aggregate", data), indent=2))
else:
    print(f"Scores file not found at {scores_path}")
    print("Check if Phase 2 completed successfully.")

## 9. Download results to your laptop

Optional — results are already in Google Drive. This downloads copies directly.

In [ ]:
from google.colab import files
from pathlib import Path

for path in [RESPONSES_PATH, SCORES_PATH]:
    p = Path(path)
    if p.exists():
        print(f"Downloading {p.name}...")
        files.download(str(p))
    else:
        print(f"⚠️  {path} not found")

## 10. Resume interrupted run (optional)

If Colab disconnected mid-run, use this cell to resume.
The cached KGs let you skip the expensive build phase.

In [ ]:
# Resume Phase 1 — skip KG build, only re-run QA on cached graphs.
# KG_DIR already points at Drive, so cached KGs survived the disconnect; just
# load them. (Builds are skipped for cached convs; QA re-runs for all.)
!python evaluation/BEAM/run_eval.py \
    --tier {TIER} \
    --max-samples {MAX_SAMPLES} \
    --max-questions {MAX_QUESTIONS} \
    --retrieval {RETRIEVAL} \
    --load-kg-dir {KG_DIR} \
    --output {RESPONSES_PATH}

print("\n=== Resume complete ===")